# Day 25 — Probability Distributions

This notebook covers Probability Mass Function (PMF), Probability Density Function (PDF), Normal (Gaussian) Distribution, Skewness, and Uniform Distribution

## Validation goals
- Understand how random variables map to probabilities via distributions
- Distinguish between discrete distributions (PMF) and continuous distributions (PDF)
- Analyze the parameters, symmetry, and empirical rule of the Normal (Gaussian) distribution
- Quantify distribution asymmetry using skewness and identify relationships between mean, median, and mode ($\text{Mode} < \text{Median} < \text{Mean}$)
- Evaluate the impact of right-skewed features on linear regression coefficient estimation and gradient descent stability

A probability distribution describes how the values of a variable are spread out across their possible range. Recognizing distribution shapes dictates whether you should use parametric models (which assume normality) or non-parametric models, how you should scale your features, and how you detect anomalies.

### Core Concepts & Deep Dive

1. **What is a Distribution?**
A distribution maps every possible value (or range of values) of a random variable to its corresponding probability of occurrence.

- Probability Mass Function (PMF): Used for discrete variables (e.g., the exact count of customer service calls: 0, 1, 2, 3). It gives the probability that a discrete random variable equals a specific value.

- Probability Density Function (PDF): Used for continuous variables (e.g., latency in milliseconds, height, temperature). Because continuous variables have an infinite number of possible precision points, the probability of any single exact value is technically zero; instead, the PDF describes the relative likelihood of values falling within a given range (the area under the curve).

2. **The Normal (Gaussian) Distribution**
Often called the "bell curve," the Normal distribution is the most important distribution in statistics and data science due to the Central Limit Theorem (which states that the sum of many independent random variables tends toward a normal distribution, regardless of the underlying distribution).

- Parameters: Completely defined by two parameters: the mean ($\mu$) controlling its center, and the standard deviation ($\sigma$) controlling its width/spread.

- Symmetry: It is perfectly symmetrical, meaning its mean, median, and mode are all equal.The 68-95-99.7 Rule: As covered in standard deviation, 68% of data falls within $\mu \pm 1\sigma$, 95% within $\mu \pm 2\sigma$, and 99.7% within $\mu \pm 3\sigma$.

3. **Skewness (Asymmetry)**
Real-world data is rarely perfectly symmetrical. Skewness measures the degree of asymmetry of a distribution around its mean:

- Right-Skewed (Positive Skew): The tail extends further to the right. The mode is less than the median, and the median is less than the mean ($\text{Mode} < \text{Median} < \text{Mean}$). Common in financial data like income, website traffic, or latency.

- Left-Skewed (Negative Skew): The tail extends further to the left. The mean is less than the median, and the median is less than the mode ($\text{Mean} < \text{Median} < \text{Mode}$). Common in metrics like exam scores where most students score high, but a few fail.

4. **The Uniform Distribution**
In a continuous uniform distribution, every value within a specified minimum ($a$) and maximum ($b$) interval has an equal and constant probability of occurring. Its PDF is flat (rectangular). It is commonly used in computer simulations, random number generation, and initializing hyperparameter search spaces.Practical Implementation: Analyzing Distributions in Python.

Here is a script generating synthetic normal and uniform datasets, calculating descriptive statistics, and measuring skewness using Pandas and NumPy.

In [1]:
import numpy as np
import pandas as pd

# 1. Generate Synthetic Distributions
np.random.seed(42)

# Normal distribution (e.g., human heights)
normal_data = np.random.normal(loc=170.0, scale=10.0, size=1000)

# Right-skewed distribution (e.g., transaction amounts using exponential distribution)
skewed_data = np.random.exponential(scale=50.0, size=1000)

df = pd.DataFrame({
    "normal_feature": normal_data,
    "skewed_feature": skewed_data
})

# 2. Compute Central Tendency & Skewness
summary_stats = pd.DataFrame({
    "Mean": df.mean(),
    "Median": df.median(),
    "Skewness": df.skew()
})

print("--- Distribution Characteristics ---")
print(summary_stats)

# 3. Detecting Skewness-Driven Discrepancies
print("\n--- Feature Analysis ---")
print(f"Normal Feature (Mean $\approx$ Median): Mean = {df['normal_feature'].mean():.2f}, Median = {df['normal_feature'].median():.2f}")
print(f"Skewed Feature (Mean > Median): Mean = {df['skewed_feature'].mean():.2f}, Median = {df['skewed_feature'].median():.2f}")

--- Distribution Characteristics ---
                      Mean      Median  Skewness
normal_feature  170.193321  170.253006  0.116976
skewed_feature   50.399297   36.296561  1.983789

--- Feature Analysis ---
Normal Feature (Mean $pprox$ Median): Mean = 170.19, Median = 170.25
Skewed Feature (Mean > Median): Mean = 50.40, Median = 36.30


### Guided Question:

If you pull summary statistics for a dataset representing user session durations on an e-commerce website and notice that the mean is 45 minutes while the median is 12 minutes:

Is this distribution left-skewed or right-skewed?

Why does this large gap between the mean and median occur?

### Answer:

1. **Skewness:** 

    This distribution is right-skewed (positively skewed).

2. **Why the gap occurs:**

    The mean is sensitive to extreme values, whereas the median is robust to them and represents the midpoint of the data. In e-commerce session durations, the vast majority of users have short visits (browsing quickly or bouncing, which keeps the median low at 12 minutes). However, a small subset of users has exceptionally long sessions—such as leaving a tab open for hours, deep-diving into product comparisons, or extensive browsing. These high-value outliers pull the arithmetic mean significantly higher (to 45 minutes), creating a long right tail.

### Guided Question:

Many linear regression algorithms and parametric statistical tests assume that features and error residuals follow a normal distribution.

What happens to model performance or coefficient estimates if you feed highly right-skewed features (such as unscaled monetary amounts) directly into a linear model without transformation?

### Answer:

Feeding highly right-skewed features (such as unscaled monetary amounts) directly into a linear regression model without transformation severely impacts both parameter estimation and predictive performance.

1. **Distorted Coefficient Estimates and High Leverage**
Ordinary Least Squares (OLS) regression minimizes the sum of squared residuals. In a right-skewed distribution, a small fraction of extreme values (the long right tail) possess disproportionately high leverage.

- The regression line is forced to "bend" toward accommodating these extreme outliers to minimize squared errors.

- As a result, the slope coefficient ($\beta$) for that feature becomes heavily skewed by the tail, failing to represent the relationship between the feature and the target variable for the vast majority (the dense bulk) of the data.

3. **Suboptimal Predictive Performance**
Because OLS optimization penalizes squared errors, the model prioritizes reducing errors on the massive monetary outliers at the expense of the dense cluster of normal data points.

- The model's predictions become overly sensitive to large values, resulting in poor predictive accuracy and high error rates (such as high RMSE) for the typical user or transaction that sits near the median.

4. **Gradient Descent Optimization Instability**
If the linear model is solved using iterative optimization algorithms like Gradient Descent rather than the closed-form normal equation:

- Features with massive scales and wide ranges cause the loss surface to become highly anisotropic (elongated, narrow "valleys").

- This ill-conditioned matrix makes gradient updates bounce back and forth inefficiently, slowing down convergence or requiring excessively small learning rates.

**Mitigation Strategies**

To prevent these issues, data scientists typically apply transformations before fitting linear models:

- Log Transformation ($\log(x + 1)$): Compresses the right tail, turning multiplicative effects into additive ones and pulling skewed distributions closer to normality.

- Feature Scaling: Standardization (Z-score) or min-max scaling helps stabilize gradient descent convergence, though it does not fix skewness itself.

- Robust Alternatives: Using tree-based models (like Random Forests or XGBoost), which are invariant to monotonic feature transformations and inherently robust to outliers.

### Guided Question:

Explain why a Probability Mass Function (PMF) can output an exact probability equal to 1 for a specific outcome in a discrete distribution, whereas a Probability Density Function (PDF) in a continuous distribution can output values greater than 1 and does not represent an exact point probability.

**Probability Mass Function (PMF) and Exact Point Probabilities**

In a discrete distribution, the random variable takes on countable, distinct values (such as integers, counts, or categorical outcomes). The Probability Mass Function, denoted as $P(X = x)$, assigns an actual, direct probability to each individual outcome.

- Why an outcome can equal 1: Because discrete outcomes are disjoint and partition the sample space into distinct buckets, a single outcome can absorb the entire probability mass. For example, in a deterministic system where a specific outcome is guaranteed to occur, $P(X = x) = 1$ (and all other possible outcomes have a probability of 0).

- The fundamental rule governing a PMF is that the sum of probabilities across all possible discrete outcomes must equal 1:$$\sum_{x} P(X = x) = 1$$

**Probability Density Function (PDF) and Density vs. Probability**

In contrast, a continuous distribution deals with an infinite, uncountable number of possible values (such as exact times, lengths, or monetary amounts).

1. **Why Point Probabilities are Zero**
In a continuous sample space, the probability of observing any single, exact point $x$ is strictly zero:$$P(X = x) = 0$$

Because there are infinitely many points between any two numbers, assigning a non-zero probability to every individual point would cause the sum of probabilities to diverge to infinity, violating the axiom that total probability must equal 1. Instead, probabilities in continuous spaces are only meaningful over intervals (e.g., $P(a \le X \le b)$), calculated via integration:$$P(a \le X \le b) = \int_{a}^{b} f(x) \,dx$$

2. **Why PDF Values Can Exceed 1**

The output of a PDF, $f(x)$, represents probability density (probability per unit of the variable), not a direct probability.While a valid probability must always fall between 0 and 1, a density is a rate. It measures how densely probability is packed around a specific point. As long as the total area under the curve across the entire sample space equals 1, the height of the curve at any specific point can be any non-negative number, including values greater than 1.$$\int_{-\infty}^{\infty} f(x) \,dx = 1$$Concrete Example:
Imagine a continuous random variable that is uniformly distributed between $0$ and $0.5$.

- The width of the interval is $0.5$.

- To ensure the total area under the density curve equals $1$ ($\text{width} \times \text{height} = 1$), the height ($f(x)$) must be:$$f(x) = \frac{1}{0.5} = 2$$

Throughout this interval, the PDF outputs a value of $2$. This value greater than 1 is mathematically valid because it is a density, meaning the actual probability of landing in a tiny window around $x$ is $2 \times \text{width}$, and the integral over any valid sub-interval remains $\le 1$.